In [53]:
import pandas as pd
from utils import *

df = local.read.csv("unit_comp_1_etl", "interim")
df

# Filter out AI first here
df['opponent_name'] = df['opponent_name'].fillna('')
df = df[~df['opponent_name'].str.contains('A.I.')].copy()
df = df[df['game_length_seconds_max_600'] >= 420].copy()
# df.opponent_name.value_counts()
df


# Can leave it raw fields.  No model specific to matchup.  Let it score it.
        
                    


,game_number,opponent_name,opponent_race,game_length_seconds_max_600,main_player_won,my_Larva,my_Drone,my_Overlord,my_Zergling,my_Queen,...,opponent_LocustMPPrecursor,opponent_Viper,opponent_Tempest,opponent_HighTemplar,my_CollapsibleRockTowerPushUnit,opponent_CollapsibleRockTowerPushUnit,opponent_HellionTank,opponent_AutoTurret,opponent_InvisibleTargetDummy,opponent_DarkTemplar
0,1,AcidCloud,Terran,600,Win,189.0,99.0,19.0,54.0,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,HORIZON,Zerg,600,Win,199.0,93.0,23.0,38.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Hydrogen,Terran,600,Win,203.0,133.0,18.0,54.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,noCCnoGG,Terran,600,Loss,199.0,92.0,26.0,72.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,IIIIIIIIIRus,Terran,600,Win,195.0,100.0,18.0,86.0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,326,llllllllllll,Protoss,601,Win,222.0,96.0,26.0,124.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324,327,OblivioN,Terran,601,Win,218.0,98.0,23.0,118.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
325,328,Sykorax,Zerg,600,Win,163.0,77.0,26.0,28.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
327,330,Aztk,Zerg,560,Win,143.0,76.0,15.0,8.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [115]:
df2 = local.read.csv("unit_costs")
support_dict = df2.set_index("Unit").to_dict()['Count for 700+ Resources']
new_support_dict = {}
for key, value in support_dict.items():
    new_support_dict["my_" + str(key)] = value
    new_support_dict["opponent_" + str(key)] = value
new_support_dict['opponent_widowmine'] = 7
new_support_dict['opponent_siegetank'] = 3
new_support_dict['opponent_voidray'] = 2
new_support_dict['opponent_warpprism'] = 3
new_support_dict['opponent_hightemplar'] = 4
new_support_dict['opponent_darktemplar'] = 3






In [116]:
df2


,Unit,Cost (Minerals / Gas),Total Cost per Unit,Count for 700+ Resources,Total Cost
0,marine,50 / 0,50,14,700
1,reaper,50 / 50,100,7,700
2,marauder,100 / 25,125,6,750
3,ghost,150 / 125,275,3,825
4,hellion,100 / 0,100,7,700
5,hellbat,100 / 0,100,7,700
6,siege tank,150 / 125,275,3,825
7,cyclone,150 / 100,250,3,750
8,thor,300 / 200,500,2,"1,000"
9,viking,150 / 75,225,4,900


In [117]:
new_support_dict['my_zergling']


28

In [118]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
ignore_columns1 = ['my_broodling', 'my_overlord', 'my_changeling', 'my_larva', 'my_drone', 'opponent_mule', 'opponent_scv', 'opponent_probe', 'opponent_drone', 'opponent_broodling', 'opponent_overlord', 'opponent_changeling', 'opponent_larva']
ignore_columns2 = ['opponent_race', 'opponent_name', 'opponent_kd8charge', 'opponent_changelingmarine', 'opponent_vikingfighter', 'opponent_changelingzerglingwings', 'opponent_nuke', 'my_forcefield', 'opponent_forcefield', 'opponent_adeptphaseshift',
                   'opponent_changelingmarineshield', 'opponent_changelingzealot','opponent_disruptorphased','opponent_interceptor','my_changelingzergling',
                   'opponent_changelingzergling', 'my_changelingzerglingwings', 'opponent_swarmhostmp', 'opponent_locustmp', 'opponent_locustmpprecursor',
                   'my_collapsiblerocktowerpushunit', 'opponent_collapsiblerocktowerpushunit', 'opponent_helliontank', 'opponent_autoturret', 'opponent_invisibletargetdummy',
                   ]

temp_df = df.query("opponent_race == 'Terran'")
temp_df.columns = [x.lower() for x in temp_df.columns]
for x in temp_df.columns:
    if ("my_" in x or "opponent_" in x) and x not in ignore_columns1 and x not in ignore_columns2:
        print(x)
        temp_df[x] = temp_df[x].apply(lambda y: 1 if y >= new_support_dict[x] else 0)

my_zergling
my_queen
my_roach
my_hydralisk
my_viper
opponent_reaper
opponent_marine
opponent_hellion
opponent_widowmine
opponent_banshee
opponent_cyclone
opponent_siegetank
opponent_liberator
opponent_zergling
opponent_queen
opponent_roach
opponent_hydralisk
opponent_ghost
my_corruptor
opponent_mutalisk
opponent_medivac
my_mutalisk
opponent_marauder
my_infestor
opponent_battlecruiser
opponent_zealot
opponent_sentry
opponent_stalker
opponent_phoenix
opponent_oracle
opponent_voidray
opponent_adept
opponent_thor
opponent_observer
opponent_immortal
opponent_disruptor
opponent_colossus
opponent_mothership
opponent_carrier
opponent_corruptor
opponent_infestor
opponent_warpprism
opponent_raven
opponent_viper
opponent_tempest
opponent_hightemplar
opponent_darktemplar


C:\Users\peter\AppData\Local\Temp\ipykernel_36740\734142396.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df[x] = temp_df[x].apply(lambda y: 1 if y >= new_support_dict[x] else 0)


In [119]:

# temp_df = df.copy()
X = temp_df[[x for x in temp_df.columns if ('my_' in x or 'opponent_' in x) and x != 'opponent_race' and x not in ignore_columns1 and x not in ignore_columns2]].copy()
y = temp_df['main_player_won'].apply(lambda x: 1 if x == 'Win' else 0)
model = DecisionTreeClassifier(max_depth=7, min_samples_leaf=5, random_state=1)
model.fit(X, y)
plt.figure(figsize=(40, 20), dpi=300)
plot_tree(model, feature_names=X.columns, class_names=[str(c) for c in model.classes_], filled=True, rounded=True, fontsize=10)
plt.tight_layout()
plt.savefig(
    "tree.pdf", format="pdf", bbox_inches="tight"
)  
plt.close()


In [21]:
df


,game_number,opponent_race,game_length_seconds_max_600,main_player_won,my_Larva,my_Drone,my_Overlord,my_Zergling,my_Queen,my_Changeling,...,opponent_LocustMPPrecursor,opponent_Viper,opponent_Tempest,opponent_HighTemplar,my_CollapsibleRockTowerPushUnit,opponent_CollapsibleRockTowerPushUnit,opponent_HellionTank,opponent_AutoTurret,opponent_InvisibleTargetDummy,opponent_DarkTemplar
0,1,Terran,600,Win,189.0,99.0,19.0,54.0,11.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Zerg,600,Win,199.0,93.0,23.0,38.0,4.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Terran,600,Win,203.0,133.0,18.0,54.0,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Terran,600,Loss,199.0,92.0,26.0,72.0,8.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Terran,600,Win,195.0,100.0,18.0,86.0,8.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,328,Zerg,600,Win,163.0,77.0,26.0,28.0,6.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
326,329,Protoss,286,Win,43.0,17.0,4.0,16.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
327,330,Zerg,560,Win,143.0,76.0,15.0,8.0,5.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
328,331,Zerg,600,Win,189.0,90.0,21.0,38.0,4.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
